<a href="https://colab.research.google.com/github/GabrielJ07/ConfiguratorAgent/blob/main/Convert_Anthropic_to_Markdown.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
from datetime import datetime

def convert_anthropic_to_md(input_file, output_file):
    """Parses Anthropic Claude conversations.json and converts it to NotebookLM-friendly Markdown."""

    if not os.path.exists(input_file):
        print(f"Error: Could not find '{input_file}'. Make sure it is in the same folder as this script.")
        return

    print(f"Reading {input_file}...")
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # NotebookLM has a 500,000 word limit per source. We use 400,000 for a safe margin.
    MAX_WORDS_PER_FILE = 400000
    base_name, ext = os.path.splitext(output_file)
    file_index = 1

    current_out_file = f"{base_name}_Part{file_index}{ext}"
    out_f = open(current_out_file, 'w', encoding='utf-8')
    out_f.write(f"# Circuittelligence - Claude Activity History (Part {file_index})\n\n")
    print(f"Writing to {current_out_file}...")

    current_word_count = 0
    convo_count = 0
    message_count = 0

    # Anthropic stores exports as a list of conversation objects
    for convo in data:
        convo_content = ""
        name = convo.get('name', 'Untitled Conversation')
        created_at = convo.get('created_at', '')

        # Format the timestamp
        try:
            # Claude timestamps usually look like "2024-01-01T12:00:00Z"
            dt = datetime.fromisoformat(created_at.replace('Z', '+00:00'))
            formatted_time = dt.strftime("%B %d, %Y at %I:%M %p")
        except ValueError:
            formatted_time = created_at

        convo_content += f"## Conversation: {name}\n"
        convo_content += f"**Started:** {formatted_time}\n\n"

        messages = convo.get('chat_messages', [])
        for msg in messages:
            # Identify if the sender was you or Claude
            sender = msg.get('sender', 'unknown').lower()
            display_sender = "User Prompt" if sender in ['user', 'human'] else "Claude Response"

            # Extract the text (Anthropic usually stores this cleanly in a "text" key)
            text = msg.get('text', '')

            # Fallback just in case they shift their schema to use the 'content' array
            if not text and 'content' in msg:
                for part in msg['content']:
                    if part.get('type') == 'text':
                        text += part.get('text', '') + "\n"

            text = text.strip()

            if text:
                convo_content += f"**{display_sender}:**\n{text}\n\n"
                message_count += 1

        convo_content += "---\n\n"

        # Check word count of this conversation
        words_in_convo = len(convo_content.split())

        # If adding this convo exceeds the limit, start a new file
        if current_word_count + words_in_convo > MAX_WORDS_PER_FILE and current_word_count > 0:
            out_f.close()
            file_index += 1
            current_out_file = f"{base_name}_Part{file_index}{ext}"
            out_f = open(current_out_file, 'w', encoding='utf-8')
            out_f.write(f"# Circuittelligence - Claude Activity History (Part {file_index})\n\n")
            print(f"File limit reached. Continuing in {current_out_file}...")
            current_word_count = 0

        out_f.write(convo_content)
        current_word_count += words_in_convo
        convo_count += 1

    out_f.close()

    print(f"Success! Converted {convo_count} conversations ({message_count} messages) across {file_index} file(s).")
    print("You can now upload these files directly into NotebookLM.")

if __name__ == "__main__":
    # Point this to the specific Anthropic file
    INPUT_FILENAME = 'conversations.json'
    OUTPUT_FILENAME = 'Circuittelligence_Claude_Activity.md'

    convert_anthropic_to_md(INPUT_FILENAME, OUTPUT_FILENAME)